In [1]:
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, hstack
from sklearn.preprocessing import LabelEncoder
from category_encoders import HashingEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

def divide_sender(list):
    name=[]
    email=[]
    for person in list:
        
        temp=person.split("<")
        if len(temp)==2:
            temp[1]=temp[1][:-1]
        else:
            if temp[0].find("@")==-1:
                temp.append("")
            else:
                temp.insert(0,"")
        name.append(temp[0])
        email.append(temp[1])
    return name,email
def divide_email(list):
    prefix=[]
    domain=[]
    for mail in list:
        temp=mail.split("@")
        if len(temp)!=2:
            prefix.append('')
            domain.append('')
        else:    
            prefix.append(temp[0])
            domain.append([temp[1]])

    return prefix,domain

folder="./email_data/"
CEAS8=pd.read_csv(folder+"CEAS_08.csv")
Enron=pd.read_csv(folder+"Enron.csv")
Ling=pd.read_csv(folder+"Ling.csv")
Nazario=pd.read_csv(folder+"Nazario.csv")
N_fraud=pd.read_csv(folder+"Nigerian_Fraud.csv")
phishing=pd.read_csv(folder+"phishing_email.csv")
spam=pd.read_csv(folder+"SpamAssasin.csv")




url_encoder=LabelEncoder()
label_encoder=LabelEncoder()
body_vectorizer=TfidfVectorizer()
subject_vectorizer=TfidfVectorizer()
name_encoder=HashingEncoder()
email_encoder=HashingEncoder()
domain_encoder=HashingEncoder()

In [2]:
CEAS=pd.DataFrame(CEAS8)
CEAS=CEAS.fillna("")
sendercopy=CEAS['sender'].copy()
name,email=divide_sender(sendercopy)
prefix,domain=divide_email(email)
subject=subject_vectorizer.fit_transform(CEAS['subject'].copy())
body=body_vectorizer.fit_transform(CEAS['body'].copy())
urls=csr_matrix((CEAS['urls'].copy().astype(int))).reshape(-1,1)
print(urls.shape)
name=name_encoder.fit_transform(name)
prefix=email_encoder.fit_transform(prefix)
domain=domain_encoder.fit_transform(domain)

print(name.shape)
print(domain.shape)
X=hstack([name,subject,body,prefix,domain,urls])
y=CEAS['label'].astype(int)

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=0)
initial_detect=LogisticRegression()
initial_detect.fit(X_train,y_train)
y_pred =initial_detect.predict(X_test)

accuracy = accuracy_score(y_test,y_pred)
conf_matrix= confusion_matrix(y_test,y_pred)
class_report=classification_report(y_test,y_pred)
print(f'Accuracy:{accuracy}')
print(conf_matrix)
print('Classification Report:')
print(class_report)





(39154, 1)
(39154, 8)
(39154, 8)
Accuracy:0.9922104456646661
[[3342   36]
 [  25 4428]]
Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.99      0.99      3378
           1       0.99      0.99      0.99      4453

    accuracy                           0.99      7831
   macro avg       0.99      0.99      0.99      7831
weighted avg       0.99      0.99      0.99      7831



In [4]:
safe=[]
blacklisted=[]

def input_email(sender,subject,body,urls):
    print('balcklisted',blacklisted)
    print(safe)
    spam_threshold=0.6
    safeexit=np.bool(False)
    badexit=np.bool(False)
    details={
        "scam?":0,
        "scamtype":"Unknown",
        "blmail":False,
        "safemail":False
    }
    name,email=divide_sender(sender)
    print(email[0])
    print(email)
    if email[0] in safe: 
        safeexit=np.bool(True)
        details["safemail"]=True   
        
    if email[0] in blacklisted: 
        badexit=np.bool(True)
        details["blmail"]=True   
    print(safeexit,badexit)
    if(safeexit):
        print("Not Scam")
        print("Reasoning: Whitelisted email",email[0])
    elif(badexit):
        details['scam?']=1
        print("Scam")
        print("Reasoning: Blacklisted email",email[0])
    else:
        prefix,domain=divide_email(email)
        subject=subject_vectorizer.transform(subject)
        body=body_vectorizer.transform(body)
        name=name_encoder.transform(name)
        prefix=email_encoder.transform(prefix)
        domain=domain_encoder.transform(domain)
        urls = np.array(urls, dtype=float).reshape(1, -1)  # shape (1, n_features)
        urls = csr_matrix(urls)
 
        print(name.shape)
        print(domain.shape)
        print(urls.shape)
        X=hstack([name,subject,body,prefix,domain,urls])
        spam_prob=initial_detect.predict_proba(X)
        details["scam?"]=spam_prob[0][1]
        if details["scam?"]>=spam_threshold:
            #RUN THE DIFFERENT MODEL TO BE ABLE TO FIND THE VALUES
            print("Likely to be spam")
            print("Reasoning: Detected scam type was",details["scamtype"] )
            pass
        else:
            print("Unlikely to be spam.")
            print("Reasoning: Predicted chance of spam was ",(details['scam?']*100),'%')
    
    return details

new_email=np.array([["jeff <helo@ching.com>"],["I need money"],["please"],[2]])
input_email(new_email[0],new_email[1],new_email[2],new_email[3])

balcklisted []
[]
helo@ching.com
['helo@ching.com']
False False
(1, 8)
(1, 8)
(1, 1)
Unlikely to be spam.
Reasoning: Predicted chance of spam was  21.88485261845336 %


{'scam?': np.float64(0.21884852618453357),
 'scamtype': 'Unknown',
 'blmail': False,
 'safemail': False}

In [ ]:

import json
import requests
import socketio

serverloc="http://localhost:3000"
safeApi="/api/data/safe"
blaclistApi="/api/data/blacklist"
sio = socketio.Client()
sio.connect(serverloc)
# io.emit('Email has been uploaded',email)
@sio.on("js_to_py")
def process_email(email):
        email=json.loads(email)
        print("recived data",email)
        detail=input_email(email['sender'],email['subject'],email['body'],email['urls'])
        sio.emit("py_to_js",detail)
@sio.on('safemail')
def changesafemail(data):
        global safe
        safe=data
        print(safe)
@sio.on('blacklisted')
def changeblacklisted(data):
        global blacklisted
        blacklisted=data
        print(blacklisted)



# keep the client running
# sio.wait()


['noreply@rncrosoft.com', 'helo@ching.com']


['support@microsoft.com', 'rafarukmanto@gmail.com', 'noreply@google.com']
recived data {'sender': ['Jeff <nope@gmail.com>'], 'subject': ['balls'], 'body': ['sadasfdfasf'], 'urls': [0]}
balcklisted ['noreply@rncrosoft.com', 'helo@ching.com']
['support@microsoft.com', 'rafarukmanto@gmail.com', 'noreply@google.com']
nope@gmail.com
['nope@gmail.com']
False False
(1, 8)
(1, 8)
(1, 1)
Likely to be spam
Reasoning: Detected scam type was Unknown
recived data {'sender': ['Jeff <nope@gmail.com>'], 'subject': ['balls'], 'body': ['sadasfdfasf'], 'urls': [0]}
balcklisted ['noreply@rncrosoft.com', 'helo@ching.com']
['support@microsoft.com', 'rafarukmanto@gmail.com', 'noreply@google.com']
nope@gmail.com
['nope@gmail.com']
False False
(1, 8)
(1, 8)
(1, 1)
Likely to be spam
Reasoning: Detected scam type was Unknown
recived data {'sender': ['Jeff <nope@gmail.com>'], 'subject': ['balls'], 'body': ['sadasfdfasf'], 'urls': [0]}
balcklisted ['noreply@rncrosoft.com', 'helo@ching.com']
['support@microsoft.com